# Teardown — Remove All Demo Resources

## Objective
Spin down all the GCP resources the demo created across the notebooks (NB1–06) — the inverse of the demo path:
- the deployed Agent Runtime (NB1 / NB3),
- the loop/rollback Pub/Sub topics + Cloud Functions + alert policies + dashboard + log-based metric (NB2 / NB4),
- the L2 CI/CD infra (NB6: WIF pool/provider, deployer SA, Artifact Registry — via `terraform destroy` of the `cicd/` folder),
- the GCS buckets (staging + per-agent artifacts), and local build scratch.

NB5 creates no GCP resources (CI is tests + GitHub Actions). The GitHub-side workflow files + Environments are removed by hand; noted below.

Scope: this tears down the notebook + `cicd/` resources. The big L1 `terraform/` config (if you used that IaC path instead of NB1–4) tears itself down with its own `terraform destroy`. The `cicd/` destroy is scoped to the `cicd/` folder and never touches `terraform/`.

Guarded: nothing runs until you set `RUN_TEARDOWN = True` in the first cell. Every destructive cell is gated on that flag, so you can read top-to-bottom before deleting anything.

Documentation: [Manage deployed agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/manage-deployed-agents)

In [ ]:
# Set to True to actually delete. Left False so this notebook is safe to open and read.
RUN_TEARDOWN = False

import os

AGENT = 'brand-guidelines-checker'
AGENT_DIR = f'../agents/{AGENT}'
os.environ.setdefault('GOOGLE_CLOUD_PROJECT', 'sandbox-401718')
PROJECT = os.environ['GOOGLE_CLOUD_PROJECT']
REGION = os.environ.get('GOOGLE_CLOUD_LOCATION', 'us-central1')

# Log-based metric name mirrors observability.tf / notebook 02: kebab -> snake.
METRIC_NAME = f"{AGENT.replace('-', '_')}_final_response_quality"

# Pub/Sub topics: the improve trigger (NB2) and the protect/rollback trigger (NB4).
LOOP_TOPIC = f'{AGENT}-loop-trigger'
ROLLBACK_TOPIC = f'{AGENT}-rollback-trigger'

# Cloud Functions: the loop subscriber (NB3) and the rollback subscriber (NB4).
LOOP_FUNCTION = f'{AGENT}-loop'
ROLLBACK_FUNCTION = f'{AGENT}-rollback'

# GCS buckets: shared staging (NB1) + per-agent artifacts/eval-runs (NB2).
STAGING_BUCKET = f'gs://{PROJECT}-agent-staging'
ARTIFACTS_BUCKET = f'gs://{PROJECT}-{AGENT}-artifacts'

# displayNames the alert policies / dashboard were created under (read from the
# specs so this stays in sync — the specs use friendly names, not the kebab id).
import yaml, json

ALERT_DISPLAY_NAME = yaml.safe_load(open(f'{AGENT_DIR}/observability/alert.yaml'))[
    'displayName'
]
CANARY_ALERT_DISPLAY_NAME = yaml.safe_load(
    open(f'{AGENT_DIR}/observability/canary_alert.yaml')
)['displayName']
DASHBOARD_DISPLAY_NAME = json.load(open(f'{AGENT_DIR}/observability/dashboard.json'))[
    'displayName'
]

print(f'project                 = {PROJECT}')
print(f'agent                   = {AGENT}')
print(f'loop-trigger topic      = {LOOP_TOPIC}')
print(f'rollback-trigger topic  = {ROLLBACK_TOPIC}')
print(f'loop function           = {LOOP_FUNCTION}')
print(f'rollback function       = {ROLLBACK_FUNCTION}')
print(f'staging bucket          = {STAGING_BUCKET}')
print(f'artifacts bucket        = {ARTIFACTS_BUCKET}')
print(f'log-based metric        = {METRIC_NAME}')
print(f'alert displayName       = {ALERT_DISPLAY_NAME!r}')
print(f'canary alert displayName= {CANARY_ALERT_DISPLAY_NAME!r}')
print(f'dashboard displayName   = {DASHBOARD_DISPLAY_NAME!r}')
print(f'\nRUN_TEARDOWN = {RUN_TEARDOWN}  (set True to enable the cells below)')

## 1. Undeploy the Agent Runtime

No `agents-cli` delete verb exists — deletion is SDK-only. `teardown_agent.py` lists the project's Agent Runtimes, matches the one whose `display_name` equals this agent's name, and deletes it (idempotent: prints what it deleted or that nothing matched). Same script Terraform's destroy-time provisioner calls.

In [ ]:
if RUN_TEARDOWN:
    !python ../deployment/teardown_agent.py --agent-dir $AGENT_DIR --project $PROJECT
else:
    print("skipped (RUN_TEARDOWN is False)")

## 2. Delete the Loop / Rollback Cloud Functions

The two subscribers that close the loop hands-off: `$AGENT-loop` (NB3, improve path) and `$AGENT-rollback` (NB4, protect path). Idempotent — `--quiet` and a tolerant exit if a function was never deployed.

In [ ]:
if RUN_TEARDOWN:
    for fn in (LOOP_FUNCTION, ROLLBACK_FUNCTION):
        print(f"deleting Cloud Function {fn}")
        !gcloud functions delete $fn --gen2 --region $REGION --project $PROJECT --quiet
else:
    print("skipped (RUN_TEARDOWN is False)")

## 3. Delete the Loop-Trigger and Rollback-Trigger Pub/Sub Topics

The `$AGENT-loop-trigger` topic the drift alert (or a scheduled offline eval) publishes to, and the `$AGENT-rollback-trigger` topic the canary breach alert publishes to.

In [ ]:
if RUN_TEARDOWN:
    for topic in (LOOP_TOPIC, ROLLBACK_TOPIC):
        print(f"deleting topic {topic}")
        !gcloud pubsub topics delete $topic --project $PROJECT --quiet
else:
    print("skipped (RUN_TEARDOWN is False)")

## 4. Delete the Alert Policies, Dashboard, and Log-Based Metric

Both alert policies (the prod drift alert and the canary breach alert) and the dashboard are created by `displayName`, so we look up each resource id by its exact name and delete every match. (Re-running NB2 / NB4 could create duplicates with the same displayName; this deletes all that match.) The GA log-based metric is deleted by its fixed name, and last, since the alerts reference it.

In [ ]:
if RUN_TEARDOWN:
    import subprocess

    def _ids(list_cmd):
        out = subprocess.run(
            list_cmd, capture_output=True, text=True, check=True
        ).stdout
        return [ln for ln in out.splitlines() if ln.strip()]

    for display_name in (ALERT_DISPLAY_NAME, CANARY_ALERT_DISPLAY_NAME):
        policy_ids = _ids(
            [
                'gcloud',
                'alpha',
                'monitoring',
                'policies',
                'list',
                f'--filter=displayName="{display_name}"',
                '--format=value(name)',
                '--project',
                PROJECT,
            ]
        )
        for pid in policy_ids:
            print(f'deleting alert policy {pid}  ({display_name})')
            subprocess.run(
                [
                    'gcloud',
                    'alpha',
                    'monitoring',
                    'policies',
                    'delete',
                    pid,
                    '--project',
                    PROJECT,
                    '--quiet',
                ],
                check=True,
            )
        if not policy_ids:
            print(f'nothing matched — alert policy {display_name!r} already gone')

    dash_ids = _ids(
        [
            'gcloud',
            'monitoring',
            'dashboards',
            'list',
            f'--filter=displayName="{DASHBOARD_DISPLAY_NAME}"',
            '--format=value(name)',
            '--project',
            PROJECT,
        ]
    )
    for did in dash_ids:
        print(f'deleting dashboard {did}')
        subprocess.run(
            [
                'gcloud',
                'monitoring',
                'dashboards',
                'delete',
                did,
                '--project',
                PROJECT,
                '--quiet',
            ],
            check=True,
        )
    if not dash_ids:
        print('nothing matched — dashboard already gone')

    # Delete the GA log-based metric last (the alerts above referenced it).
    print(f'deleting log-based metric {METRIC_NAME}')
    subprocess.run(
        [
            'gcloud',
            'logging',
            'metrics',
            'delete',
            METRIC_NAME,
            '--project',
            PROJECT,
            '--quiet',
        ]
    )
else:
    print('skipped (RUN_TEARDOWN is False)')

## 5. Tear Down the L2 CI/CD Infra (NB6)

`terraform destroy` the `cicd/` config — removes the WIF pool/provider, the deployer SA + roles, and the Artifact Registry repo (and any images in it). Then deletes the generated `pipeline.yml` so PRs stop triggering. Scoped to `cicd/` only — the big `terraform/` L1 config is untouched.

Note: do these by hand (not GCP) — the `staging`/`production` Environments in repo Settings, and (if you want) the committed `.github/workflows/{ci,cd}.yml`. Destroying a WIF pool soft-deletes it (30-day retention before the id frees up).

In [ ]:
if RUN_TEARDOWN:
    # Scoped to the cicd/ folder: WIF pool/provider, deployer SA + roles, Artifact Registry.
    !terraform -chdir=../cicd destroy -auto-approve
    import pathlib
    p = pathlib.Path('../.github/workflows/pipeline.yml')
    if p.exists():
        p.unlink(); print('removed generated .github/workflows/pipeline.yml (PR trigger gone)')
else:
    print("skipped (RUN_TEARDOWN is False)")

## 6. Delete the GCS Buckets

The shared staging bucket (NB1: packaged source + `agent-payloads`) and the per-agent artifacts bucket (NB2: managed-eval outputs), if present. Recursive + idempotent.

In [ ]:
if RUN_TEARDOWN:
    for b in (STAGING_BUCKET, ARTIFACTS_BUCKET):
        print(f"deleting bucket {b}")
        !gcloud storage rm -r $b --project $PROJECT 2>/dev/null || echo "  ({b} not found / already gone)"
else:
    print("skipped (RUN_TEARDOWN is False)")

## 7. Manual / Out-of-Scope Items

- **Online monitor (Preview):** console-driven, no scripted delete. Remove it by hand: Agent Platform > Agents > Evaluation > Online monitors.
- **Service accounts / IAM:** anything created by `agents-cli infra single-project` is owned by that command's own Terraform and is torn down separately — this notebook doesn't touch it.
- **Local results:** `agents/<name>/results/` is gitignored scratch; the cell below clears it.

In [ ]:
print(
    f"Reminder: remove the Preview online monitor for {AGENT} in the Agent Platform console — it is not torn down automatically."
)

if RUN_TEARDOWN:
    !rm -rf $AGENT_DIR/results ../_serving_build _serving_build ../cicd/.terraform
    print("cleared local scratch: results/, _serving_build/, cicd/.terraform/")
else:
    print("skipped local results cleanup (RUN_TEARDOWN is False)")